# Customer support KB

In [1]:
%%capture
!pip install -q datasets sentence-transformers simlar simlar-engine ipywidgets

## Load the knowledge base

We use [`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
— ~27,000 real customer-support utterances, each with a canonical `response`, a
`category`, and an `intent`. `instruction` is the
question a customer asked; `response` is the answer an agent would give.

In [ ]:
# optional
from huggingface_hub import login
login(token="<your_token>")

In [3]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")

QUESTIONS = [str(q) for q in ds["instruction"]] # what we match against
ANSWERS   = [str(a) for a in ds["response"]] # the KB answer to surface
INTENTS   = [str(i) for i in ds["intent"]]
IDS       = [f"kb_{i}" for i in range(len(QUESTIONS))]

id_to_q = dict(zip(IDS, QUESTIONS))
id_to_a = dict(zip(IDS, ANSWERS))
id_to_intent = dict(zip(IDS, INTENTS))

print(f"Loaded {len(QUESTIONS)} KB entries")
print(f"Sample question: {QUESTIONS[0][:80]}")
print(f"Sample answer:   {ANSWERS[0][:80]}")

Loaded 26872 KB entries
Sample question: question about cancelling order {{Order Number}}
Sample answer:   I've understood you have a question regarding canceling order {{Order Number}}, 


## Embed

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(QUESTIONS, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

## Using simlar

`search(query, k)` returns ranked `SearchResult`s (`.rank`, `.id`, `.score`). The engine ranks by proximity in embedding space — **results are sorted best-first by `rank`**, and its `score` is a distance, so a **lower `score` is a closer match**. We answer the ticket with the top hit's KB answer.

In [5]:
from simlar import SimlarEngine

kb = SimlarEngine()
kb.add(ids=IDS[:400], vectors=vectors[:400])   # seed with the first 400 entries
print(f"KB size: {kb.size}  trained: {kb.is_trained}")

KB size: 400  trained: True


In [6]:
def match(ticket, k=3):
    qv = model.encode([ticket], normalize_embeddings=True).astype(np.float32)
    hits = kb.search(qv, k=k)
    print(f"Ticket: {ticket!r}\n")
    for r in hits:
        print(f"  rank={r.rank}  score={r.score:.2f}  intent={id_to_intent[r.id]}")
        print(f"    matched Q: {id_to_q[r.id][:70]}")
    return hits

hits = match("how do I get my money back for an order I cancelled?")
print("\nSuggested answer:\n", id_to_a[hits[0].id][:200])

Ticket: 'how do I get my money back for an order I cancelled?'

  rank=0  score=0.28  intent=cancel_order
    matched Q: I purchased some item, help canceling one of the orders
  rank=1  score=0.24  intent=cancel_order
    matched Q: I have got to cancel purchase {{Order Number}}, help me
  rank=2  score=0.24  intent=cancel_order
    matched Q: I have got to cancel purchase {{Order Number}}, how do I do it?

Suggested answer:
 I understand your request for assistance with canceling one of the orders for the item you purchased. I apologize for any inconvenience, and I'm here to guide you through the process.

To cancel an or


### Grow the KB without rebuilding

In [7]:
kb.add(ids=IDS[400:], vectors=vectors[400:])
print(f"KB size after incremental add: {kb.size}")

KB size after incremental add: 26872


### Update and delete

| Method | When you use it |
|---|---|
| `update(ids, vectors)` | An article was reworded — replace its stored vector in place (size unchanged) |
| `delete(ids)` | A policy was retired — remove the article so it stops surfacing |

In [8]:
# An article was reworded: re-embed just that one entry and update in place.
reworded = "I want to request a refund for a purchase I already cancelled"
new_vec = model.encode([reworded], normalize_embeddings=True).astype(np.float32)
kb.update(ids=[IDS[0]], vectors=new_vec)
print(f"After update: size {kb.size} (unchanged — update is in place)")

# Two articles were retired: delete them so they never surface again.
kb.delete(ids=[IDS[5], IDS[6]])
print(f"After delete: size {kb.size}")

After update: size 26872 (unchanged — update is in place)
After delete: size 26870
